In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, cohen_kappa_score, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Day 4 ke saved patches load karna
npz_path = "/content/drive/MyDrive/TerraSpectra/data/patches/terraspectra_patches.npz"
data = np.load(npz_path)

def prepare_loader(X, y, batch_size=64, shuffle=False):
    X_t = np.transpose(X, (0, 3, 1, 2))   # (N, 30, 15, 15)
    X_t = np.expand_dims(X_t, axis=1)     # (N, 1, 30, 15, 15)
    ds = TensorDataset(torch.tensor(X_t, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = prepare_loader(data["X_train"], data["y_train"], batch_size=64, shuffle=True)
val_loader   = prepare_loader(data["X_val"],   data["y_val"],   batch_size=64, shuffle=False)
test_loader  = prepare_loader(data["X_test"],  data["y_test"],  batch_size=64, shuffle=False)

num_classes = len(np.unique(data["y_train"]))
print(f"Data Loaded! Classes: {num_classes} | Train batches: {len(train_loader)}")

Using device: cuda
Data Loaded! Classes: 16 | Train batches: 113


In [2]:
class Spectral3DViTFusion(nn.Module):
    def __init__(self, num_classes=16, embed_dim=128, num_heads=4, num_layers=2, dropout=0.2):
        super().__init__()

        # 1. 3D-CNN Backbone (Local Spatial-Spectral Extraction)
        # Input: (B, 1, 30, 15, 15) -> Output: (B, 32, 18, 9, 9)
        self.conv3d = nn.Sequential(
            nn.Conv3d(1, 8, kernel_size=(7, 3, 3)),
            nn.BatchNorm3d(8),
            nn.GELU(),
            nn.Conv3d(8, 16, kernel_size=(5, 3, 3)),
            nn.BatchNorm3d(16),
            nn.GELU(),
            nn.Conv3d(16, 32, kernel_size=(3, 3, 3)),
            nn.BatchNorm3d(32),
            nn.GELU(),
        )

        # 32 channels * 18 spectral depth = 576 features per spatial position
        # Spatial grid is 9x9 = 81 tokens
        num_tokens = 9 * 9
        self.token_proj = nn.Linear(32 * 18, embed_dim)

        # 2. ViT Tokens: [CLS] token + Learnable Positional Embeddings
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, num_tokens + 1, embed_dim) * 0.02)
        self.pos_drop = nn.Dropout(dropout)

        # 3. Multi-Head Self-Attention Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 2,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 4. Classification Head on [CLS] token
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # 3D-CNN feature extraction: (B, 32, 18, 9, 9)
        x = self.conv3d(x)
        b, c, d, h, w = x.shape

        # Reshape into sequence of 81 spatial-spectral tokens: (B, 81, 576)
        x = x.view(b, c * d, h * w).transpose(1, 2)
        x = self.token_proj(x)  # (B, 81, embed_dim)

        # Prepend [CLS] token & add positional encoding: (B, 82, embed_dim)
        cls_tokens = self.cls_token.expand(b, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = self.pos_drop(x + self.pos_embed)

        # Multi-Head Self-Attention Fusion
        x = self.transformer(x)

        # Classify using fused [CLS] representation
        cls_out = self.norm(x[:, 0])
        return self.head(cls_out)

model = Spectral3DViTFusion(num_classes=num_classes).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"3D-CNN + ViT Fusion Model Ready! Total Parameters: {total_params:,}")

/tmp/ipykernel_628/3004561937.py:39: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


3D-CNN + ViT Fusion Model Ready! Total Parameters: 379,248


In [3]:
SAVE_DIR = "/content/drive/MyDrive/TerraSpectra/data/artifacts"
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = f"{SAVE_DIR}/vit_fusion_indian_pines.pth"

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

best_val_acc = 0.0

for epoch in range(1, 51):
    model.train()
    correct, total = 0, 0
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        outputs = model(X_b)
        loss = criterion(outputs, y_b)
        loss.backward()
        optimizer.step()

        correct += (outputs.argmax(1) == y_b).sum().item()
        total += y_b.size(0)
    train_acc = (correct / total) * 100

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            preds = model(X_b).argmax(1)
            val_correct += (preds == y_b).sum().item()
            val_total += y_b.size(0)
    val_acc = (val_correct / val_total) * 100
    scheduler.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/50 | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | Best Val: {best_val_acc:.2f}%")

# --- Final Test Evaluation ---
model.load_state_dict(torch.load(MODEL_PATH))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for X_b, y_b in test_loader:
        X_b = X_b.to(device)
        all_preds.extend(model(X_b).argmax(1).cpu().numpy())
        all_targets.extend(y_b.numpy())

test_oa = accuracy_score(all_targets, all_preds) * 100
kappa = cohen_kappa_score(all_targets, all_preds)

print(f"\n=== DAY 7 ViT FUSION TEST RESULTS ===")
print(f"Test Overall Accuracy (OA): {test_oa:.2f}%")
print(f"Cohen's Kappa Score:        {kappa:.4f}")

metrics_day7 = {
    "model": "Spectral3DViTFusion (3D-CNN + Multi-Head Self-Attention ViT)",
    "dataset": "Indian Pines (15x15x30 patches)",
    "test_overall_accuracy": round(test_oa, 2),
    "cohen_kappa": round(float(kappa), 4),
    "num_classes": int(num_classes),
    "test_samples": len(all_targets)
}

with open(f"{SAVE_DIR}/metrics_day7.json", "w") as f:
    json.dump(metrics_day7, f, indent=2)

print("Saved vit_fusion_indian_pines.pth and metrics_day7.json!")

Epoch 01/50 | Train Acc: 82.17% | Val Acc: 95.64% | Best Val: 95.64%
Epoch 05/50 | Train Acc: 98.80% | Val Acc: 98.57% | Best Val: 98.83%
Epoch 10/50 | Train Acc: 99.23% | Val Acc: 99.74% | Best Val: 99.74%
Epoch 15/50 | Train Acc: 99.93% | Val Acc: 99.80% | Best Val: 99.80%
Epoch 20/50 | Train Acc: 99.99% | Val Acc: 99.80% | Best Val: 99.87%
Epoch 25/50 | Train Acc: 100.00% | Val Acc: 99.87% | Best Val: 99.93%
Epoch 30/50 | Train Acc: 99.94% | Val Acc: 99.87% | Best Val: 99.93%
Epoch 35/50 | Train Acc: 99.97% | Val Acc: 99.87% | Best Val: 99.93%
Epoch 40/50 | Train Acc: 100.00% | Val Acc: 99.87% | Best Val: 99.93%
Epoch 45/50 | Train Acc: 100.00% | Val Acc: 99.87% | Best Val: 99.93%
Epoch 50/50 | Train Acc: 100.00% | Val Acc: 99.87% | Best Val: 99.93%

=== DAY 7 ViT FUSION TEST RESULTS ===
Test Overall Accuracy (OA): 99.93%
Cohen's Kappa Score:        0.9993
Saved vit_fusion_indian_pines.pth and metrics_day7.json!
